In [1]:
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier
classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
signal, fs =torchaudio.load('output1.wav')
real_embeddings = classifier.encode_batch(signal)
test_signal , fs = torchaudio.load('part_031_s2.wav')
test_emb = classifier.encode_batch(test_signal)



/home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: module 'torchaudio' has no attribute 'list_audio_backends'

In [ ]:
import torch
import torch.nn.functional as F

def implement_centroid_detection(ref_embeddings, test_embedding, threshold=0.25):
    """
    ref_embeddings: tensor from encode_batch, shape (1, 1, 192) or (N, 1, 192)
    test_embedding: tensor from encode_batch, shape (1, 1, 192)
    """
    # Flatten to 2D: (N, 192)
    ref_embeddings = ref_embeddings.squeeze()   # handles (1,1,192) → (192,) or (N,1,192) → (N,192)
    test_embedding = test_embedding.squeeze()   # (192,)

    # If single ref embedding is already 1D (192,), use as-is
    # If multiple ref embeddings (N, 192), compute centroid
    if ref_embeddings.dim() == 2:
        ref_embeddings = torch.mean(ref_embeddings, dim=0)  # (192,)

    # Normalize
    centroid = F.normalize(ref_embeddings, p=2, dim=0)      # (192,)
    test_embedding = F.normalize(test_embedding, p=2, dim=0) # (192,)

    # Cosine similarity via dot product of normalized vectors
    score_value = torch.dot(centroid, test_embedding).item()
    result = score_value > threshold

    return score_value, result


# Usage
score, is_same_speaker = implement_centroid_detection(real_embeddings, test_emb)
print(f"Cosine similarity: {score:.4f}")
print(f"Same speaker: {is_same_speaker}")